# IPL Cricket - Predicting Batsman Runs Using Regression

**Course:** Predictive Analytics  
**Group Members:**
- Member 1: Adithyan Biju - Problem Statement and Feature Engineering
- Member 2: Fidal Govind
- Member 3: Archana Das

**Dataset:** IPL Complete Dataset 2008-2020 (Kaggle)  
**Models:** Simple Linear Regression | Multiple Linear Regression | Logistic Regression

---
## STAGE 1 - Problem Definition and Literature Review
**Led by: Member 1**

### 1.1 Problem Statement

Cricket is one of the most data-rich sports in the world, yet predicting individual player performance remains a challenging task due to the many variables involved. This project aims to predict the number of runs a batsman will score in an IPL match using historical performance data.

**What we are predicting:** Runs scored by a batsman in an IPL match  
**Why it matters:** Useful for team selection, fantasy cricket, and match strategy  
**Dataset:** IPL ball-by-ball data from 2008 to 2020

### 1.2 Research Objectives

1. To build a Simple Linear Regression model as a baseline predictor
2. To improve prediction accuracy using Multiple Linear Regression with 4 features
3. To classify batsman performance as Flop / Average / Excellent using Logistic Regression
4. To compare all three models and identify the best performing approach

### 1.3 Research Questions

- Can a batsman's recent form predict their next match score?
- Which features (career average, recent form, strike rate, venue) have the most influence?
- Can we reliably classify performance categories using Logistic Regression?



### 1.4 Research Gap

*(Member 1 - Write 2-3 sentences about what existing papers have NOT done that your project does)* 
Most existing studies focus on match-level prediction rather than individual player performance. This project addresses that gap by building a player-level prediction pipeline that combines regression and classification in one framework.

---
## STAGE 2 - Data Collection and Understanding
**Led by: Member 2**

In [9]:
!pip install shap streamlit

In [10]:
# Member 1 - Import all required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    r2_score, mean_squared_error, mean_absolute_error,
    accuracy_score, classification_report, confusion_matrix
)

print('All libraries imported successfully!')

All libraries imported successfully!


In [11]:
# Member 2 - Load the datasets
# Make sure deliveries.csv and matches.csv are in the same folder
deliveries = pd.read_csv('deliveries.csv')
matches = pd.read_csv('matches.csv')

print('Deliveries dataset shape:', deliveries.shape)
print('Matches dataset shape:', matches.shape)

Deliveries dataset shape: (260920, 17)
Matches dataset shape: (1095, 20)


In [12]:
# Member 2 - Explore the deliveries dataset
print('=== DELIVERIES COLUMNS ===')
print(deliveries.columns.tolist())

print('\n=== FIRST 5 ROWS ===')
print(deliveries.head())

print('\n=== DATA TYPES ===')
print(deliveries.dtypes)

print('\n=== MISSING VALUES ===')
print(deliveries.isnull().sum())

=== DELIVERIES COLUMNS ===
['match_id', 'inning', 'batting_team', 'bowling_team', 'over', 'ball', 'batter', 'bowler', 'non_striker', 'batsman_runs', 'extra_runs', 'total_runs', 'extras_type', 'is_wicket', 'player_dismissed', 'dismissal_kind', 'fielder']

=== FIRST 5 ROWS ===
   match_id  inning           batting_team                 bowling_team  over  \
0    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   
1    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   
2    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   
3    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   
4    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   

   ball       batter   bowler  non_striker  batsman_runs  extra_runs  \
0     1   SC Ganguly  P Kumar  BB McCullum             0           1   
1     2  BB McCullum  P Kumar   SC Ganguly             0           0   
2     3  BB McCullum 

In [13]:
# Member 2 - Explore the matches dataset
print('=== MATCHES COLUMNS ===')
print(matches.columns.tolist())

print('\n=== FIRST 5 ROWS ===')
print(matches.head())

print('\n=== MISSING VALUES ===')
print(matches.isnull().sum())

print('\n=== TOTAL MATCHES ===')
print(len(matches))

print('\n=== SEASONS AVAILABLE ===')
print(sorted(matches['season'].unique()))

=== MATCHES COLUMNS ===
['id', 'season', 'city', 'date', 'match_type', 'player_of_match', 'venue', 'team1', 'team2', 'toss_winner', 'toss_decision', 'winner', 'result', 'result_margin', 'target_runs', 'target_overs', 'super_over', 'method', 'umpire1', 'umpire2']

=== FIRST 5 ROWS ===
       id   season        city        date match_type player_of_match  \
0  335982  2007/08   Bangalore  2008-04-18     League     BB McCullum   
1  335983  2007/08  Chandigarh  2008-04-19     League      MEK Hussey   
2  335984  2007/08       Delhi  2008-04-19     League     MF Maharoof   
3  335985  2007/08      Mumbai  2008-04-20     League      MV Boucher   
4  335986  2007/08     Kolkata  2008-04-20     League       DJ Hussey   

                                        venue                        team1  \
0                       M Chinnaswamy Stadium  Royal Challengers Bangalore   
1  Punjab Cricket Association Stadium, Mohali              Kings XI Punjab   
2                            Feroz Shah Ko

---
## STAGE 3 - Data Preprocessing and Cleaning
**Led by: Member 3**

In [14]:
# Member 3 - Calculate runs per batsman per match
# Check if 'deliveries' and 'matches' DataFrames exist, if not, load them
if 'deliveries' not in globals() or 'matches' not in globals():
    import pandas as pd
    try:
        deliveries = pd.read_csv('deliveries.csv')
        matches = pd.read_csv('matches.csv')
        print('Deliveries and Matches datasets loaded (conditionally).')
    except FileNotFoundError:
        print("Error: 'deliveries.csv' or 'matches.csv' not found. Please ensure they are in the correct directory.")

batsman_match = (
    deliveries
    .groupby(['match_id', 'batter'])['batsman_runs']
    .sum()
    .reset_index()
    .rename(columns={'batter': 'batsman', 'batsman_runs': 'runs_scored'})
)

print('Batsman-match table shape:', batsman_match.shape)
print(batsman_match.head(10))

Deliveries and Matches datasets loaded (conditionally).
Batsman-match table shape: (16515, 3)
   match_id          batsman  runs_scored
0    335982        AA Noffke            9
1    335982          B Akhil            0
2    335982      BB McCullum          158
3    335982         CL White            6
4    335982        DJ Hussey           12
5    335982        JH Kallis            8
6    335982       MV Boucher            7
7    335982  Mohammad Hafeez            5
8    335982          P Kumar           18
9    335982         R Dravid            2


In [15]:
# Member 3 - Calculate balls faced and strike rate
balls_faced_to_add = (
    deliveries[deliveries['extras_type'] != 'wides']
    .groupby(['match_id', 'batter'])['ball']
    .count()
    .reset_index()
    .rename(columns={'batter': 'batsman', 'ball': 'balls_faced'})
)

# Ensure no 'balls_faced' column exists in batsman_match before merging to avoid suffixes
# This handles cases where the cell might be re-run or kernel state is inconsistent
for col in ['balls_faced', 'balls_faced_x', 'balls_faced_y']:
    if col in batsman_match.columns:
        batsman_match.drop(columns=[col], inplace=True)

batsman_match = batsman_match.merge(balls_faced_to_add, on=['match_id', 'batsman'], how='left')
batsman_match['strike_rate'] = (batsman_match['runs_scored'] / batsman_match['balls_faced'] * 100).round(2)
batsman_match['balls_faced'] = batsman_match['balls_faced'].fillna(1)
batsman_match['strike_rate'] = batsman_match['strike_rate'].fillna(0)

print('Strike rate added.')
print(batsman_match.head())

Strike rate added.
   match_id      batsman  runs_scored  balls_faced  strike_rate
0    335982    AA Noffke            9         10.0        90.00
1    335982      B Akhil            0          2.0         0.00
2    335982  BB McCullum          158         73.0       216.44
3    335982     CL White            6         10.0        60.00
4    335982    DJ Hussey           12         12.0       100.00


In [16]:
# Member 3 - Merge with match information (venue, teams)
match_info = matches[['id', 'venue', 'team1', 'team2', 'winner']].copy()
match_info.rename(columns={'id': 'match_id'}, inplace=True)

batsman_match = batsman_match.merge(match_info, on='match_id', how='left')

# Find the opposition team for each batsman
bat_team = deliveries.groupby(['match_id', 'batter'])['batting_team'].first().reset_index()
bat_team.rename(columns={'batter': 'batsman'}, inplace=True) # Rename 'batter' to 'batsman'
batsman_match = batsman_match.merge(bat_team, on=['match_id', 'batsman'], how='left')
batsman_match['opposition'] = batsman_match.apply(
    lambda r: r['team2'] if r['batting_team'] == r['team1'] else r['team1'], axis=1
)

# Sort by match_id so rolling averages are in time order
batsman_match = batsman_match.sort_values('match_id').reset_index(drop=True)

print('Match info merged.')
print(batsman_match.shape)

Match info merged.
(16515, 11)


In [17]:
# Member 3 - Remove duplicates and check data quality
print('Duplicates before:', batsman_match.duplicated().sum())
batsman_match.drop_duplicates(inplace=True)
print('Duplicates after:', batsman_match.duplicated().sum())

print('\nNull values per column:')
print(batsman_match.isnull().sum())

print('\nTotal records:', len(batsman_match))

Duplicates before: 0
Duplicates after: 0

Null values per column:
match_id         0
batsman          0
runs_scored      0
balls_faced      0
strike_rate      0
venue            0
team1            0
team2            0
winner          40
batting_team     0
opposition       0
dtype: int64

Total records: 16515


### 3.1 Feature Engineering: Historical Performance
**Led by: Member 1**
To build our predictive models, we need to generate features that capture the batsman's historical form prior to the current match. We will engineer:
- **Career Average**: The cumulative average runs scored by the batsman up to the previous match.
- **Recent Form**: The average runs scored in the batsman's last 3 matches.

In [18]:
# Member 1 - Feature Engineering
def calculate_historical_features(df):
    # Sort chronologically by match_id for each batsman
    df = df.sort_values(by=['batsman', 'match_id'])
    
    # Shift by 1 to prevent data leakage (using current match runs to predict current match runs)
    df['career_avg'] = df.groupby('batsman')['runs_scored'].transform(lambda x: x.shift(1).expanding().mean())
    df['recent_form'] = df.groupby('batsman')['runs_scored'].transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
    
    # Fill NaNs (e.g., debut matches) with 0
    df['career_avg'] = df['career_avg'].fillna(0)
    df['recent_form'] = df['recent_form'].fillna(0)
    return df

batsman_features = calculate_historical_features(batsman_match.copy())
print('Feature Engineering complete. Previewing new columns:')
print(batsman_features[['batsman', 'match_id', 'runs_scored', 'career_avg', 'recent_form']].head())

---
## STAGE 4 - Modeling and Evaluation
**Led by: All Members Collaboratively**
We will now build and evaluate our three models as per our research objectives.

In [19]:
# Train-Test Split
X = batsman_features[['career_avg', 'recent_form', 'balls_faced', 'strike_rate']]
y = batsman_features['runs_scored']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Model 1: Simple Linear Regression (Baseline predicting strictly from recent form)
slr = LinearRegression()
slr.fit(X_train[['recent_form']], y_train)
slr_preds = slr.predict(X_test[['recent_form']])

# Model 2: Multiple Linear Regression
mlr = LinearRegression()
mlr.fit(X_train, y_train)
mlr_preds = mlr.predict(X_test)

print('=== Regression Evaluation ===')
print(f'Simple Linear Regression Mean Absolute Error: {mean_absolute_error(y_test, slr_preds):.2f} runs')
print(f'Multiple Linear Regression Mean Absolute Error: {mean_absolute_error(y_test, mlr_preds):.2f} runs')
print(f'Multiple Linear Regression R-squared Score: {r2_score(y_test, mlr_preds):.4f}')

In [20]:
# Model 3: Logistic Regression (Classification of Performance Category)
def categorize_performance(runs):
    if runs < 15:
        return 'Flop'
    elif runs <= 40:
        return 'Average'
    else:
        return 'Excellent'

y_train_class = y_train.apply(categorize_performance)
y_test_class = y_test.apply(categorize_performance)

# Standardize features for Logistic Regression convergence and performance
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train_scaled, y_train_class)
logreg_preds = logreg.predict(X_test_scaled)

print('\n=== Logistic Regression Evaluation ===')
print(classification_report(y_test_class, logreg_preds))